In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import f1_score, roc_auc_score, recall_score, precision_score
from tqdm import trange
from IPython.display import clear_output

# 🔹 Correzione della Normalizzazione
scaler = MinMaxScaler(feature_range=(0.1, 10))  # Range più ampio per evitare valori troppo piccoli

# 📌 Creazione delle Sliding Windows
def create_windows(df, window_size=10, step=1):
    grouped = df.groupby("case:concept:name")
    sequences = []
    labels = []

    for _, group in grouped:
        seq = list(zip(group["activity_encoded"], group["duration_scaled"]))

        for i in range(0, len(seq) - window_size + 1, step):
            window = seq[i : i + window_size]
            label = group["Property 1"].iloc[-1]  # La label è quella della traccia intera

            sequences.append(window)
            labels.append(label)

    return sequences, np.array(labels)

# Creazione delle finestre per train e test
window_size = 10
train_sequences, train_labels = create_windows(train_df, window_size=window_size, step=1)
test_sequences, test_labels = create_windows(test_df, window_size=window_size, step=1)

# Padding per uniformare le dimensioni
max_seq_length = window_size  # Ora è fisso
train_padded = pad_sequences(train_sequences, maxlen=max_seq_length, padding="post", dtype="float32")
test_padded = pad_sequences(test_sequences, maxlen=max_seq_length, padding="post", dtype="float32")

# Dataset Class
class TraceDataset(Dataset):
    def __init__(self, sequences, labels):
        self.sequences = torch.tensor(sequences, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.float32)
    
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]

# 🔍 Debug: Controllo dati di input PRIMA del training
print("🔍 CHECK DATI DI INPUT")
for i in range(5):
    print(f"Sequenza {i}: {train_padded[i]}")
    print(f"Label: {train_labels[i]}")
    print("-" * 80)

# Load Data
batch_size = 32
train_dataset = TraceDataset(train_padded, train_labels)
test_dataset = TraceDataset(test_padded, test_labels)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# 📌 Modello Migliorato
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=0.3)
        self.batch_norm = nn.BatchNorm1d(hidden_size)  # Aiuta la stabilizzazione del training
        self.fc = nn.Linear(hidden_size, output_size)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        last_out = lstm_out[:, -1, :]
        last_out = self.batch_norm(last_out)
        return self.sigmoid(self.fc(last_out))

# 🚀 Parametri migliorati
input_size = 2  # (activity_id, duration)
hidden_size = 256  # Aumentiamo la capacità del modello
num_layers = 3  # Più profondità
output_size = 1  

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = LSTMModel(input_size, hidden_size, num_layers, output_size).to(device)

# 🔧 Loss & Optimizer aggiornati
criterion = nn.BCELoss()
optimizer = optim.AdamW(model.parameters(), lr=0.0005)

# 📊 Training Loop
epochs = 10
train_losses, train_accs, train_f1s = [], [], []
val_losses, val_accs, val_f1s = [], [], []

epoch_bar = trange(epochs, ncols=100)
for epoch in epoch_bar:
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    all_train_preds, all_train_true = [], []

    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        outputs = model(batch_x).squeeze()
        predicted = (outputs > 0.5).float()
        
        all_train_preds.extend(predicted.detach().cpu().numpy())
        all_train_true.extend(batch_y.detach().cpu().numpy())

        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * batch_x.size(0)
        correct += (predicted == batch_y).sum().item()
        total += batch_y.size(0)

    # 🔍 Controlliamo la perdita e l'accuratezza
    train_loss = running_loss / len(train_dataset)
    train_acc = correct / total
    train_f1 = f1_score(all_train_true, all_train_preds, average='binary')

    train_losses.append(train_loss)
    train_accs.append(train_acc)
    train_f1s.append(train_f1)

    # 🔎 Validazione
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    all_val_preds, all_val_true = [], []

    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            outputs = model(batch_x).squeeze()
            predicted = (outputs > 0.5).float()

            all_val_preds.extend(predicted.detach().cpu().numpy())
            all_val_true.extend(batch_y.detach().cpu().numpy())

            val_loss += criterion(outputs, batch_y).item() * batch_x.size(0)
            val_correct += (predicted == batch_y).sum().item()
            val_total += batch_y.size(0)

    # 📊 Metriche di validazione
    val_loss /= len(test_dataset)
    val_acc = val_correct / val_total
    val_f1 = f1_score(all_val_true, all_val_preds, average='binary')

    val_losses.append(val_loss)
    val_accs.append(val_acc)
    val_f1s.append(val_f1)

    # 🏆 Stampa dei risultati
    epoch_bar.set_description(
        f"Epoch {epoch+1}/{epochs}: T_Loss {train_loss:.4f}, T_Acc {train_acc:.4f}, T_F1 {train_f1:.4f} | "
        f"V_Loss {val_loss:.4f}, V_Acc {val_acc:.4f}, V_F1 {val_f1:.4f}")

# 🔍 Valutazione Finale sul Test Set
print("\n🔍 TEST FINALE SUL TEST SET")

model.eval()
test_correct, test_total = 0, 0
all_test_preds, all_test_true = [], []

with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        outputs = model(batch_x).squeeze()
        predicted = (outputs > 0.5).float()

        all_test_preds.extend(predicted.detach().cpu().numpy())
        all_test_true.extend(batch_y.detach().cpu().numpy())

test_acc = test_correct / test_total
test_f1 = f1_score(all_test_true, all_test_preds, average='binary')

print(f"\n📊 Test Accuracy: {test_acc:.4f}, F1 Score: {test_f1:.4f}")

# 🔥 Salvataggio del modello finale
torch.save(model.state_dict(), "lstm_model.pth")
print("✅ Model saved as lstm_model.pth")

NameError: name 'train_df' is not defined

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.nn.utils.rnn import pad_sequence
from tqdm import trange, tqdm
import matplotlib.pyplot as plt
from IPython.display import clear_output
from sklearn.metrics import f1_score
import time
import os

# Set up device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device in use: {device}")

def load_trace_data(csv_file, prefix_extraction=True, min_prefix_length=2, max_trace_length=50):
    """
    Loads the CSV, sorts by case and start_time, and constructs sequences (prefixes)
    assigning property 1: 1 if the difference between A_Submitted and O_Sent (if present)
    is less than or equal to 48 hours (172800 sec), otherwise 0.
    """
    df = pd.read_csv(csv_file)
    df['start_time'] = pd.to_datetime(df['start_time'])
    df = df.sort_values(['case:concept:name', 'start_time'])

    traces, labels = [], []
    activity_to_idx = {}
    idx_counter = 1

    for case_id, group in tqdm(df.groupby('case:concept:name'), desc="Extracting prefixes"):
        submitted_time = None
        sent_time = None
        seq = []
        prev_time = None

        for _, row in group.iterrows():
            event_name = row['concept:name']
            timestamp = row['start_time']

            if event_name not in activity_to_idx:
                activity_to_idx[event_name] = idx_counter
                idx_counter += 1

            if event_name == 'A_Submitted':
                submitted_time = timestamp
            if event_name == 'O_Sent' and submitted_time is not None and sent_time is None:
                sent_time = timestamp

            # Calculate the time delta between consecutive events
            delta = 0 if prev_time is None else (timestamp - prev_time).total_seconds()
            prev_time = timestamp
            seq.append([activity_to_idx[event_name], delta])

            # Extract prefix if minimum length is reached
            if prefix_extraction and len(seq) >= min_prefix_length:
                # Calculate the difference between A_Submitted and O_Sent (if available)
                diff_time = (sent_time - submitted_time).total_seconds() if (submitted_time and sent_time) else -1
                # If the difference is positive and <= 48 hours (172800 sec), label is 1, otherwise 0
                prop_value = 1.0 if 0 < diff_time <= 172800 else 0.0
                traces.append(torch.tensor(seq[:], dtype=torch.float32))
                labels.append(prop_value)

    traces_tensor = pad_sequence(traces, batch_first=True, padding_value=0)
    labels_tensor = torch.tensor(labels, dtype=torch.float32)
    return traces_tensor, labels_tensor, activity_to_idx

# Load train and test data
train_traces, train_labels, act2idx = load_trace_data('train.csv', prefix_extraction=True, min_prefix_length=2)
test_traces, test_labels, _ = load_trace_data('test.csv', prefix_extraction=True, min_prefix_length=2)

train_dataset = TensorDataset(train_traces, train_labels)
test_dataset = TensorDataset(test_traces, test_labels)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Define the simple attention mechanism
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super(Attention, self).__init__()
        self.attn = nn.Linear(hidden_dim, 1)

    def forward(self, lstm_out, mask):
        # lstm_out: (batch, seq_len, hidden_dim)
        scores = self.attn(lstm_out).squeeze(-1)  # (batch, seq_len)
        scores = scores.masked_fill(mask == 0, float('-inf'))
        attn_weights = torch.softmax(scores, dim=1).unsqueeze(1)  # (batch, 1, seq_len)
        weighted_sum = torch.bmm(attn_weights, lstm_out).squeeze(1)  # (batch, hidden_dim)
        return weighted_sum

# Define the model integrating embedding, time transformation, bidirectional LSTM, and attention
class TraceClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, lstm_hidden):
        super(TraceClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.time_linear = nn.Linear(1, 8)
        # LSTM input: concatenation of embedding and time features
        self.lstm = nn.LSTM(embed_dim + 8, lstm_hidden, batch_first=True, bidirectional=True)
        self.attention = Attention(lstm_hidden * 2)
        self.fc = nn.Linear(lstm_hidden * 2, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # x: (batch, seq_len, 2) --> 1: activity, 2: time delta
        act = x[:, :, 0].long()
        time = x[:, :, 1].unsqueeze(-1)
        emb = self.embedding(act)  # (batch, seq_len, embed_dim)
        time_feat = self.time_linear(time)  # (batch, seq_len, 8)
        combined = torch.cat([emb, time_feat], dim=2)  # (batch, seq_len, embed_dim+8)
        lstm_out, _ = self.lstm(combined)  # (batch, seq_len, lstm_hidden*2)
        # Create a mask to ignore padding (0 for activity 0)
        mask = (act != 0).float()
        attn_out = self.attention(lstm_out, mask)  # (batch, lstm_hidden*2)
        out = self.fc(attn_out)
        return self.sigmoid(out).squeeze()

vocab_size = max(act2idx.values()) + 1
model = TraceClassifier(vocab_size, embed_dim=16, lstm_hidden=64)
model.to(device)

# Compute class weights on the training set
# train_labels is a tensor of shape (N,)
class_counts = torch.tensor([(train_labels == 0).sum(), (train_labels == 1).sum()], dtype=torch.float32)
# Each class weight is balanced relative to the dataset size
class_weights = (1.0 / class_counts) * (len(train_labels) / 2.0)
class_weights = class_weights.to(device)

optimizer = optim.Adam(model.parameters(), lr=0.001)
epochs = 5  
progress_bar_ncols = 100

train_losses = []
train_accs = []
train_f1s = []
val_losses = []
val_accs = []
val_f1s = []

epoch_bar = trange(epochs, ncols=progress_bar_ncols)
start_train_time = time.time()
epoch_times = []

for epoch in epoch_bar:
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    all_train_preds = []
    all_train_true = []
    
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        outputs = model(batch_x)
        # Create a tensor of weights for the batch: if label is 1, use weight for 1, otherwise for 0
        weights = torch.where(batch_y == 1, class_weights[1], class_weights[0])
        loss = nn.functional.binary_cross_entropy(outputs, batch_y, weight=weights)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * batch_x.size(0)
        predicted = (outputs > 0.5).float()
        correct += (predicted == batch_y).sum().item()
        total += torch.numel(batch_y)
        all_train_preds.extend(predicted.cpu().numpy())
        all_train_true.extend(batch_y.cpu().numpy())
    
    epoch_loss = running_loss / len(train_dataset)
    epoch_acc = correct / total
    epoch_f1 = f1_score(all_train_true, all_train_preds, average='binary')
    train_losses.append(epoch_loss)
    train_accs.append(epoch_acc)
    train_f1s.append(epoch_f1)
    
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    all_val_preds = []
    all_val_true = []
    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            outputs = model(batch_x)
            # Compute weights for the batch
            weights = torch.where(batch_y == 1, class_weights[1], class_weights[0])
            loss = nn.functional.binary_cross_entropy(outputs, batch_y, weight=weights)
            val_loss += loss.item() * batch_x.size(0)
            predicted = (outputs > 0.5).float()
            val_correct += (predicted == batch_y).sum().item()
            val_total += torch.numel(batch_y)
            all_val_preds.extend(predicted.cpu().numpy())
            all_val_true.extend(batch_y.cpu().numpy())
    val_loss /= len(test_dataset)
    val_acc = val_correct / val_total
    val_f1 = f1_score(all_val_true, all_val_preds, average='binary')
    val_losses.append(val_loss)
    val_accs.append(val_acc)
    val_f1s.append(val_f1)
    
    epoch_times.append(time.time() - start_train_time)
    avg_epoch_time = sum(epoch_times) / len(epoch_times)
    remaining_time = avg_epoch_time * (epochs - (epoch + 1))
    
    epoch_bar.set_description(
        f"Epoch {epoch+1}/{epochs}: T_Loss {epoch_loss:.4f}, T_Acc {epoch_acc:.4f}, T_F1 {epoch_f1:.4f} | "
        f"V_Loss {val_loss:.4f}, V_Acc {val_acc:.4f}, V_F1 {val_f1:.4f} | ETA: {remaining_time:.2f} sec"
    )
    
    clear_output(wait=True)
    epochs_range = range(1, epoch+2)
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 4))
    ax1.plot(epochs_range, train_losses, label="Train Loss")
    ax1.plot(epochs_range, val_losses, label="Validation Loss")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.set_title("Loss per Epoch")
    ax1.set_xticks(list(epochs_range))
    ax1.legend()
    
    ax2.plot(epochs_range, train_accs, label="Train Accuracy")
    ax2.plot(epochs_range, val_accs, label="Validation Accuracy")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Accuracy")
    ax2.set_title("Accuracy per Epoch")
    ax2.set_xticks(list(epochs_range))
    ax2.legend()
    
    ax3.plot(epochs_range, train_f1s, label="Train F1")
    ax3.plot(epochs_range, val_f1s, label="Validation F1")
    ax3.set_xlabel("Epoch")
    ax3.set_ylabel("F1 Score")
    ax3.set_title("F1 Score per Epoch")
    ax3.set_xticks(list(epochs_range))
    ax3.legend()
    
    plt.tight_layout()
    plt.show()

total_train_time = time.time() - start_train_time
print(f"\nTraining completed in {total_train_time:.2f} seconds.")

model.eval()
test_loss = 0.0
test_correct = 0
test_total = 0
all_test_preds = []
all_test_true = []
with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        outputs = model(batch_x)
        weights = torch.where(batch_y == 1, class_weights[1], class_weights[0])
        loss = nn.functional.binary_cross_entropy(outputs, batch_y, weight=weights)
        test_loss += loss.item() * batch_x.size(0)
        predicted = (outputs > 0.5).float()
        test_correct += (predicted == batch_y).sum().item()
        test_total += torch.numel(batch_y)
        all_test_preds.extend(predicted.cpu().numpy())
        all_test_true.extend(batch_y.cpu().numpy())

test_loss /= len(test_dataset)
test_acc = test_correct / test_total
test_f1 = f1_score(all_test_true, all_test_preds, average='binary')
print(f"\nTest Loss: {test_loss:.4f}, Test Accuracy: {test_acc:.4f}, Test F1: {test_f1:.4f}")

all_test_true_np = np.array(all_test_true)
all_test_preds_np = np.array(all_test_preds)
correct_0 = np.sum((all_test_true_np == 0) & (all_test_preds_np == 0))
correct_1 = np.sum((all_test_true_np == 1) & (all_test_preds_np == 1))
total_0 = np.sum(all_test_true_np == 0)
total_1 = np.sum(all_test_true_np == 1)
print(f"Class 0: Correct predictions {correct_0} out of {total_0}")
print(f"Class 1: Correct predictions {correct_1} out of {total_1}")

torch.save(model.state_dict(), 'trace_classifier.pt')
print("Model saved as trace_classifier.pt")

In [2]:
import random
import numpy as np
import matplotlib.pyplot as plt
import torch

# Definiamo il device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Supponiamo che il dataset di test e il modello siano già definiti.
# test_dataset deve restituire per ogni indice una tupla (trace, true_label)
# dove trace è un tensore di shape [seq_len, 2] e la prima colonna contiene gli indici delle attività,
# la seconda i delta temporali (in secondi).

# --- Dopo aver raggruppato le attività dai dati di training, creiamo il vocabolario ---
# Ad esempio, se hai già definito train_activities come lista di liste:
# train_activities = [['A_Submitted', 'B_Review', 'O_Sent', ...], ...]
vocab = {}
for seq in train_activities:  # train_activities deve essere definita precedentemente
    for token in seq:
        if token not in vocab:
            vocab[token] = len(vocab) + 1  # inizio da 1; 0 è riservato al padding
vocab_size = len(vocab) + 1  # includiamo il token 0

# Creiamo il mapping inverso: da indice ad attività
idx2act = {v: k for k, v in vocab.items()}

# Selezione casuale di una traccia dal test_dataset con durata >= 5 minuti (300 secondi)
min_duration = 300  # 5 minuti

selected_trace = None
selected_true_label = None
selected_cumulative_times = None

while True:
    rand_idx = random.randint(0, len(test_dataset) - 1)
    trace, true_label = test_dataset[rand_idx]  # trace: [seq_len, 2]; true_label: tensore o scalare
    # La prima colonna: indici delle attività, la seconda: delta temporali (in secondi)
    deltas = trace[:, 1].cpu().numpy()
    
    # Calcoliamo i tempi cumulati
    cumulative_times = np.cumsum(deltas)
    if len(cumulative_times) > 0:
        cumulative_times[0] = 0.0

    # Calcoliamo il numero di eventi reali (ignorando il padding, dove indice attività = 0)
    actual_length = int((trace[:, 0] != 0).sum().item())

    # Se la traccia non è vuota, calcoliamo la durata totale
    if actual_length > 0:
        total_duration = cumulative_times[actual_length - 1]
    else:
        total_duration = 0.0

    # Verifichiamo che la traccia abbia almeno 2 eventi e duri almeno 5 minuti
    if actual_length >= 2 and total_duration >= min_duration:
        selected_trace = trace
        selected_true_label = true_label
        selected_cumulative_times = cumulative_times
        break

print(f"Selected trace index: {rand_idx}, Real label: {int(selected_true_label.item())}")

print("\nSequence of events with their actual times (ignoring padding):")
for i in range(actual_length):
    act_idx = int(selected_trace[i, 0].item())
    event_time = selected_cumulative_times[i]
    act_name = idx2act.get(act_idx, f"Activity {act_idx}")
    hrs = int(event_time // 3600)
    mins = int((event_time % 3600) // 60)
    secs = int(event_time % 60)
    print(f"Event {i+1}: {act_name}, Time: {event_time:.2f} sec ({hrs}h {mins}m {secs}s)")

print(f"\nTotal case duration: {total_duration:.2f} sec "
      f"({int(total_duration//3600)}h {int((total_duration%3600)//60)}m {int(total_duration%60)}s)")

# Predizioni continue nel tempo: da T=0 a T = total_duration + 600 sec
max_T = total_duration + 600
time_points = np.arange(0, max_T + 60, 60)
predictions = []

# Individuiamo l'evento "A_Submitted" per calcolare il tempo trascorso da quell'evento
idx_submitted = next(
    (i for i, idx in enumerate(selected_trace[:, 0].tolist())
     if idx != 0 and idx2act.get(int(idx), '') == 'A_Submitted'),
    None
)
submitted_time = selected_cumulative_times[idx_submitted] if idx_submitted is not None else 0.0

# Logica per fissare la predizione se l'evento "O_Sent" è stato osservato entro 24 ore
o_sent_seen = False  
final_pred = None    

print("\nPredictions at each time point:")

for T in time_points:
    # Troviamo quanti eventi reali (non padding) rientrano entro T
    prefix_length = np.searchsorted(selected_cumulative_times[:actual_length], T, side='right')
    prefix_length = max(prefix_length, 1)  # almeno 1 evento

    # Costruiamo il prefisso: usiamo solo gli indici delle attività (la parte temporale non viene usata dal modello)
    if T <= total_duration:
        prefix_activities = selected_trace[:prefix_length, 0].clone()
    else:
        prefix_activities = selected_trace[:actual_length, 0].clone()

    cumulative_time_since_submitted = max(0, T - submitted_time)

    # Prepara l'input per il modello (forma [1, seq_len])
    prefix_activities_input = prefix_activities.unsqueeze(0).long().to(device)
    
    # Se non abbiamo ancora fissato la predizione finale, controlliamo la presenza di "O_Sent" entro 24h da "A_Submitted"
    if not o_sent_seen:
        for act_idx in prefix_activities.tolist():
            if act_idx != 0 and idx2act.get(int(act_idx), '') == 'O_Sent':
                if cumulative_time_since_submitted <= 86400:
                    final_pred = 1.0
                    o_sent_seen = True
                break
        if cumulative_time_since_submitted > 86400 and not o_sent_seen:
            final_pred = 0.0

    with torch.no_grad():
        if final_pred is not None:
            pred_prefix = final_pred
        else:
            # Otteniamo la predizione all'ultimo timestep del prefisso
            out = model(prefix_activities_input)  # shape: [1, seq_len, 1]
            pred_prefix = out[0, -1, 0].item()

    predictions.append(pred_prefix)
    valid_acts = [idx2act.get(int(a_idx), f"Activity {int(a_idx)}")
                  for a_idx in prefix_activities.tolist() if a_idx != 0]
    print(f"At {T/60:.1f} min: events = {valid_acts}, prediction = {pred_prefix:.4f}")

plt.figure(figsize=(10, 5))
plt.plot(time_points / 60, predictions, marker='o', linestyle='-')
plt.xlabel("Time (minutes)")
plt.ylabel("Predicted Probability")
plt.title("Model Predicted Probability Over Continuous Time")
plt.grid(True)
plt.show()

NameError: name 'train_activities' is not defined